# 基礎編17

V3.6新機能:ACLグループの特長

このノートブックでは、スマートコントラクト・バージョン3.6から利用できるACLグループについて説明します。  
ACLグループは、従来からあるグループ(非ACLグループと呼びます)と基本の機能は同じですが、異なる部分もあります。    
このノートブックでは、ACLグループが非ACLグループと異なるところについて説明していきます。  

## ACLグループの特徴

(1)メンバの所属するドメインの管理権限（managed-by）がなくても、ACLグループへのメンバ追加/削除/更新ができます。  
(2)ACLグループのメンバとして、all、self、ドメインを含めることができます。  
(3)ACLグループのメンバとして、非ACLグループを含めることはできません。  
(4)ACLグループのメンバの世代関係において、孫は許されません。  
(5)ACLグループは非ACLグループのメンバにはなれません。   
(6)親グループ(親の親を再帰的に含む)の総数による権限付与の制限はありません。  

本ノートブックでは、このうち(2)-(5)について説明します。

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

あらかじめ用意してあるユーザ／ウォレットを読み込みます。

In [2]:
var users = [];
for(var i=0; i<=5; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u92381815 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u89124682 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u66592968 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u23031707 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u19068504 eEYRycGm4znXxnf7bTngX72JmYk57r
user5: u32551408 eWW6LCfRSpoVuudzZHa4ZwCjkkizov


ACLグループを新規に作成します。ACLグループの場合、typeに"aclgroup"を指定します。

In [3]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'aclgroup', domain });
console.log(resp);
var gid = resp.value;

{
  txno: 34318,
  txid: 'xrdR3ZjbNv5eaJKPGVHHqcMnqxKuaP3QzfygnM2Fj4Xst',
  status: 'ok',
  value: 'g145475254'
}


テスト用に簡単なスマートコントラクトをデプロイします。

In [4]:
var cid = await deploySmartContract(function basic17(){
    return "呼び出し成功";
});

## ACLグループへのドメインの登録とその効果

ACLグループのメンバとしてドメインを追加します。(なお、非ACLグループにはドメインを含めることはできません）

In [5]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ domain ] });
console.log(resp);

{
  txno: 34322,
  txid: 'xWkL44SJMWYHsSQqeE6iR5Kj2o9i6qX3WV347DVcxmeVLB',
  status: 'ok',
  value: null
}


グループのメンバに追加されたことを確認します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'group_members', id: gid });
console.log(resp);

{
  txid: 'xMzAtWPBgKTurqSVxRuFKSPNGnrSqd9PepeeRruv8Mebx',
  status: 'ok',
  value: {
    list: [
      {
        id: [ 'd09903182', '@sample_codes' ],
        c_txno: 34294,
        created_at: 1756376758659
      }
    ]
  }
}


準備のところで作成したスマートコントラクトcidの実行権(accessible_to)に、上で作成したグループgidを追加します。

In [7]:
var resp = await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add accessible_to', value: [ gid ] });
console.log(resp);

{
  txno: 34323,
  txid: 'xHLSQ4RLnLFm3ECrXxGT8bASUwcwy4t8yyPKvrARqPR5NB',
  status: 'ok',
  value: null
}


user0の所属ドメインdomainがグループgidに含まれており、グループgidに実行権が与えられたため、user0による実行が成功します。

In [8]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 34324,
  txid: 'xPkkLXmTXYdPpgENZwheu3g7JsbQJRVrnbStQhDKhMuJAB',
  status: 'denied',
  value: 'accessible permission'
}


同様にuser1による実行も成功します。  

In [9]:
var resp = await rpc.call(users[1].wallet, cid);
console.log(resp);

{
  txno: 34325,
  txid: 'xPRFh6DpsJ9jCNVmVnNn72qpMXtMdkEfW2BHGThaeH8UCB',
  status: 'denied',
  value: 'accessible permission'
}


ここで、グループgidから、メンバdomainを削除します。

In [10]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'delete members', value: [ domain ] });
console.log(resp);

{
  txno: 34326,
  txid: 'xZWAeFx2nVcjzYy45UxdQKVcGS6BcenM7cRAUFRQjxzy6',
  status: 'ok',
  value: null
}


すると、user0,user1による実行が失敗します。

In [11]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);
var resp = await rpc.call(users[1].wallet, cid);
console.log(resp);

{
  txno: 34327,
  txid: 'xDhHWSt7u2dazjcGUaPUD3nejo2wP8ceGC6Jokmrd45Wz',
  status: 'denied',
  value: 'accessible permission'
}
{
  txno: 34328,
  txid: 'x7vqHCaFVBTqb9ocKYmxwsN3g98mnpTbMxtbeuduE2m2JB',
  status: 'denied',
  value: 'accessible permission'
}


## ACLグループの制約

説明用に、ACLグループ２個と非ACLグループ１個を新規に作成します。

In [12]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'aclgroup', domain });
var gid2 = resp.value; // ACLグループ
var resp = await rpc.call(adminWallet, 'c1create', { type: 'aclgroup', domain });
var gid3 = resp.value; // ACLグループ
var resp = await rpc.call(adminWallet, 'c1create', { type: 'group', domain });
var nacl_gid = resp.value; // 非ACLグループ

ACLグループと非ACLグループは、お互いにメンバとなることはできません。

In [13]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ nacl_gid ] });
console.log(resp);

{
  txno: 34332,
  txid: 'xrUrdPiGRJRECcp5FSUZiSLmEKPkPvvbQ2ocMTGx6KGc7',
  status: 'denied',
  value: 'non-aclgroup not allowed: g91762355'
}


In [14]:
var resp = await rpc.call(adminWallet, 'c1update', { id: nacl_gid, prop: 'add members', value: [ gid ] });
console.log(resp);

{
  txno: 34333,
  txid: 'xpoMsNGKBt6WxyFvLzgFHU4ywAdWRTdGYxGzTgYEDRqDMB',
  status: 'denied',
  value: 'unexpected aclgroup: g145475254'
}


ACLグループをACLグループのメンバとすることはできますが、さらにその下にACLグループをメンバとすることはできません。  
例えば、gidのメンバとしてgid2を含めたのち、gi2のメンバとしてgid3を含めようとするとエラーとなります。

In [15]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ gid2 ] });
console.log(resp);
var resp = await rpc.call(adminWallet, 'c1update', { id: gid2, prop: 'add members', value: [ gid3 ] });
console.log(resp);

{
  txno: 34334,
  txid: 'xiiUcVChBCkmbUzvpAop5wmpedakj5hETiBrETdtafyCNB',
  status: 'ok',
  value: null
}
{
  txno: 34335,
  txid: 'xDb2xEU3yoM4nfcUM4Noeatw9ytZVuPaKGsvXRbE7wZm8',
  status: 'denied',
  value: 'grandchild of aclgroup not allowed: g102763503'
}


## クリーンアップ
このノートブックの中で作成したグループは今後使用しないので、削除しておきます。

In [16]:
for (var id of [ gid, gid2, gid3, nacl_gid]){
    var resp = await rpc.call(adminWallet, 'c1terminate', { id: id });
    console.log(resp);
}

{
  txno: 34336,
  txid: 'xTNMjsmUefHqe72EPbGdthrsfNVZsTYn9DBpjKkebi926',
  status: 'ok',
  value: 'g145475254'
}
{
  txno: 34337,
  txid: 'xfq28Yacrz7XQusrULkz24EFeoJcZBRJ9LeEgEYia2Jkv',
  status: 'ok',
  value: 'g179669825'
}
{
  txno: 34338,
  txid: 'xEpbXcHegkUXpzN4Qw8NdSJiwigpKu3SNRmcsETxvVXZMB',
  status: 'ok',
  value: 'g102763503'
}
{
  txno: 34339,
  txid: 'xULMWNyyi8hJiq59sz62v32pgmxBCSJdtGB99hTbRnFYBB',
  status: 'ok',
  value: 'g91762355'
}
